In [1]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

data = pd.read_csv("Data/sonar.csv", header=None)
data

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032,R
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044,R
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078,R
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117,R
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
203,0.0187,0.0346,0.0168,0.0177,0.0393,0.1630,0.2028,0.1694,0.2328,0.2684,...,0.0116,0.0098,0.0199,0.0033,0.0101,0.0065,0.0115,0.0193,0.0157,M
204,0.0323,0.0101,0.0298,0.0564,0.0760,0.0958,0.0990,0.1018,0.1030,0.2154,...,0.0061,0.0093,0.0135,0.0063,0.0063,0.0034,0.0032,0.0062,0.0067,M
205,0.0522,0.0437,0.0180,0.0292,0.0351,0.1171,0.1257,0.1178,0.1258,0.2529,...,0.0160,0.0029,0.0051,0.0062,0.0089,0.0140,0.0138,0.0077,0.0031,M
206,0.0303,0.0353,0.0490,0.0608,0.0167,0.1354,0.1465,0.1123,0.1945,0.2354,...,0.0086,0.0046,0.0126,0.0036,0.0035,0.0034,0.0079,0.0036,0.0048,M


In [2]:
y = data[60]
X = data.drop(columns=60).copy()

In [3]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

----

# LDA+GBM and PCA+GBM

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Any, Dict

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# ---------------------------------------------------------------------
# DuoBoostCV — evaluates only LDA+GB and PCA+GB
# ---------------------------------------------------------------------
@dataclass
class DuoBoostCV:
    """Compare LDA+GradientBoost vs PCA+GradientBoost using cross-validation.

    Parameters
    ----------
    seed : int, default=42
        Seed used both for CV (if not specified) and for the models.
    """

    seed: int = 42
    _models: Dict[str, Any] = field(init=False, repr=False)

    # -----------------------------------------------------------
    # Build models based on the number of features
    # -----------------------------------------------------------
    def _build_models(self, n_features: int) -> Dict[str, Any]:
        half_components = max(1, n_features // 2)
        return {
            "PCA + Gradient Boost": Pipeline(
                [
                    ("pca", PCA(n_components=half_components, random_state=self.seed)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
            "LDA + Gradient Boost": Pipeline(
                [
                    ("lda", LDA(n_components=None)),
                    ("gb", GradientBoostingClassifier(random_state=self.seed)),
                ]
            ),
        }

    # -----------------------------------------------------------
    # Public API — accepts X, y + all cross_validate parameters
    # -----------------------------------------------------------
    def fit(self, X, y, **cv_kwargs) -> pd.DataFrame:
        X = np.asarray(X)
        y = np.asarray(y)

        cv_kwargs = cv_kwargs.copy()
        cv_kwargs.setdefault("scoring", "accuracy")

        self._models = self._build_models(X.shape[1])
        scores: Dict[str, Dict[str, float]] = {}

        for name, model in self._models.items():
            cv_res = cross_validate(model, X, y, **cv_kwargs)

            # <<< dynamic fix >>>
            metric_key = next(k for k in cv_res if k.startswith("test_"))
            metric_name = metric_key.replace("test_", "")

            scores[name] = {
                f"mean_{metric_name}": cv_res[metric_key].mean(),
                f"std_{metric_name}":  cv_res[metric_key].std(),
            }

        self.results_ = pd.DataFrame(scores).T.sort_values(
            by=f"mean_{metric_name}", ascending=False
        )
        return self.results_
    
    # convenience
    def __call__(self, X, y, **cv_kwargs) -> pd.DataFrame:
        return self.fit(X, y, **cv_kwargs)

    def __repr__(self) -> str:  # pragma: no cover
        rep = f"{self.__class__.__name__}(seed={self.seed})"
        if hasattr(self, "results_"):
            rep += f"\nResults:\n{self.results_.round(4)}"
        return rep
    
trainer = DuoBoostCV(seed=42)

# same arguments you’d pass to cross_validate
results = trainer(
    X, y_encoded,
    cv=10,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

print(results)


                      mean_score  std_score
PCA + Gradient Boost    0.644762   0.132197
LDA + Gradient Boost    0.624524   0.150238


----

# GBM

In [17]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Define the model
gb_clf = GradientBoostingClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42
)

# Perform 5-fold cross-validation and compute accuracy
cv_scores = cross_val_score(gb_clf, X, y_encoded, cv=5, scoring='accuracy')

# Print mean and standard deviation of accuracy
print(f"Cross-validated accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

Cross-validated accuracy: 0.6259 ± 0.0871


-------

# GBM

In [9]:
X = X.to_numpy()

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score


# ---------- utility ----------
def softmax(F):
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)


# ---------- model ----------
class CustomMulticlassGradientBoostingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

        # attributes that will hold the learned state (with trailing “_”)
        self.estimators_ = None
        self.lda_transforms_ = None
        self.initial_logit_ = None
        self.classes_ = None

    # -------------------------- training --------------------------
    def fit(self, X, y):
        # reset state
        self.estimators_ = []
        self.lda_transforms_ = []
        self.initial_logit_ = None
        self.classes_ = None

        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]

        prior = np.clip(np.mean(one_hot_y, axis=0), 1e-5, 1 - 1e-5)
        self.initial_logit_ = np.log(prior)

        F = np.tile(self.initial_logit_, (n_samples, 1))

        for m in range(self.n_estimators):
            # 1) LDA
            if m == 0:
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, y)
            else:
                p = softmax(F)
                resid = one_hot_y - p
                labels = np.argmax(resid, axis=1)
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, labels)

            self.lda_transforms_.append(lda)

            # 2) updated residuals
            p = softmax(F)
            resid = one_hot_y - p

            # 3) one tree per class
            estims_m = []
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth)
                reg.fit(X_lda, resid[:, k])
                estims_m.append(reg)
                F[:, k] += self.learning_rate * reg.predict(X_lda)

            self.estimators_.append(estims_m)

        return self

    # -------------------------- inference --------------------------
    def _decision_function(self, X):
        n_samples = X.shape[0]
        F = np.tile(self.initial_logit_, (n_samples, 1))
        for lda, estims_m in zip(self.lda_transforms_, self.estimators_):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estims_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return F

    def predict_proba(self, X):
        return softmax(self._decision_function(X))

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ---------- cross-validation ----------
def cross_validate_model(X, y, model, cv=10):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs = []

    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        model_fold = clone(model)          # now clone works
        model_fold.fit(X_tr, y_tr)
        accs.append(accuracy_score(y_te, model_fold.predict(X_te)))

    return np.mean(accs), np.std(accs)


# ---------- example ----------
model = CustomMulticlassGradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7
)

mean_acc, std_acc = cross_validate_model(X, y_encoded, model, cv=10)
print(f"Mean accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

Accuratezza media: 0.7310 ± 0.0823
